# Cuaderno A: Tendencias Ampliadas (AOD, T2M y PM10 Red MÓNICA)

**Proyecto Microclima · NASA Space Apps Challenge × GAMC Cochabamba**

Este cuaderno amplía el conjunto de indicadores climáticos del Enfoque 1:
1. **LST (Superficie)**: MODIS MOD11A2 (ya en la base 2001–2019).
2. **NDVI (Vegetación)**: MODIS MOD13Q1 (ya en la base 2001–2025).
3. **AOD (Aerosoles/Calidad atmosférica)**: MODIS MCD19A2 MAIAC 1 km (2001–2025).
4. **Temperatura del aire a 2 m**: NASA MERRA-2 (~50 km) (2001–2025).
5. **PM10 en estaciones de superficie**: Red MÓNICA (Kanata y SEMAPA, jul. 2022–oct. 2026).

> **Archivos requeridos en `/content/`:**
> - `distritos_indicadores_ambientales_2024.geojson`
> - `tendencias_distritos.json`
> - `red_monica.kanata (1).csv`
> - `red_monica.semapa (1).csv`

### Celda A1 · Preparación del entorno y funciones estadísticas

In [ ]:
!pip install -q earthengine-api pymannkendall
import ee, json, time, pandas as pd, numpy as np, pymannkendall as mk
from scipy import stats

ee.Authenticate()
ee.Initialize(project='microclimas-cochabamba')

gj = json.load(open('/content/distritos_indicadores_ambientales_2024.geojson', encoding='utf-8'))
CAMPO_ID, CAMPO_NOMBRE = 'codigo_distrito', 'nombre'
distritos = ee.FeatureCollection(gj).select([CAMPO_ID, CAMPO_NOMBRE])
AÑO_INI, AÑO_FIN = 2001, 2025

# Misma función de tendencias que usamos en microclima.ipynb
def analizar(serie):
    s = serie.set_index('fecha').asfreq('MS')
    v = s.iloc[:, 0]
    r = mk.seasonal_test(v.values, period=12)
    anom = v - v.groupby(v.index.month).transform('mean')
    ok = anom.dropna()
    t = (ok.index.year + (ok.index.month - 1) / 12).values
    lr = stats.linregress(t, ok.values)
    tcrit = stats.t.ppf(0.975, len(ok) - 2)
    return {
        'sen_pendiente_anual': round(float(r.slope), 5),
        'mk_p': round(float(r.p), 4),
        'mk_tendencia': r.trend,
        'significativa': bool(r.p < 0.05),
        'lineal_pendiente_anual': round(float(lr.slope), 5),
        'lineal_p': round(float(lr.pvalue), 4),
        'ic95': [round(float(lr.slope - tcrit * lr.stderr), 5), round(float(lr.slope + tcrit * lr.stderr), 5)],
        'meses_validos': int(v.notna().sum())
    }

### Celda A2 · AOD y temperatura del aire por distrito y mes (Google Earth Engine)
Un cálculo por año con reintentos para evitar límites de cuota (10–20 minutos).

In [ ]:
aod_col = ee.ImageCollection('MODIS/061/MCD19A2_GRANULES').select('Optical_Depth_055')
t2m_col = ee.ImageCollection('NASA/GSFC/MERRA/slv/2').select('T2M')
vacia = ee.Image.constant(0).updateMask(ee.Image.constant(0))

def media_mes(col, ini, fin):
    c = col.filterDate(ini, fin)
    return ee.Image(ee.Algorithms.If(c.size().gt(0), c.mean(), vacia))

def imagen_anual(y):
    b = []
    for m in range(1, 13):
        ini = ee.Date.fromYMD(y, m, 1)
        fin = ini.advance(1, 'month')
        b.append(media_mes(aod_col, ini, fin).multiply(0.001).rename(f'aod_{m:02d}'))
        b.append(media_mes(t2m_col, ini, fin).subtract(273.15).rename(f't2m_{m:02d}'))
    return ee.Image.cat(b)

filas = []
for y in range(AÑO_INI, AÑO_FIN + 1):
    for intento in range(6):
        try:
            fc = (imagen_anual(y)
                  .reduceRegions(collection=distritos, reducer=ee.Reducer.mean(), scale=1000, tileScale=4)
                  .map(lambda f: f.setGeometry(None)))
            feats = fc.getInfo()['features']
            break
        except Exception as e:
            print(y, 'reintento en', 20 * (intento + 1), 's')
            time.sleep(20 * (intento + 1))
    for f in feats:
        p = f['properties']
        for m in range(1, 13):
            filas.append({
                CAMPO_ID: p[CAMPO_ID],
                CAMPO_NOMBRE: p[CAMPO_NOMBRE],
                'fecha': f'{y}-{m:02d}',
                'aod': p.get(f'aod_{m:02d}'),
                't2m': p.get(f't2m_{m:02d}')
            })
    print('Listo', y)
    time.sleep(2)

df2 = pd.DataFrame(filas)
df2['fecha'] = pd.to_datetime(df2['fecha'])
df2.to_csv('series_aod_t2m_distritos.csv', index=False)
print(df2.describe())

### Celda A3 · PM10 mensual de las estaciones Red MÓNICA (GAMC)

In [ ]:
RUTAS = {'Kanata': '/content/red_monica.kanata (1).csv', 'SEMAPA': '/content/red_monica.semapa (1).csv'}

def pm10_mensual(ruta):
    d = pd.read_csv(ruta, parse_dates=['fecha_medicion'])
    d = d[(d['pm10'] > 0) & (d['pm10'] < 1000)].set_index('fecha_medicion')['pm10']   # limpieza
    dia = d.resample('D').agg(['mean', 'count'])
    dia = dia[dia['count'] >= 18]['mean']            # día válido: al menos 18 de 48 mediciones
    mes = dia.resample('MS').agg(['mean', 'count'])
    return mes[mes['count'] >= 15]['mean']           # mes válido: al menos 15 días

estaciones = []
for nombre, ruta in RUTAS.items():
    s = pm10_mensual(ruta)
    t = analizar(s.rename('pm10').rename_axis('fecha').reset_index())
    print(nombre, t)
    estaciones.append({
        'nombre': nombre,
        'serie': [{'fecha': f.strftime('%Y-%m'), 'pm10': round(v, 1)} for f, v in s.items()],
        'tendencia': t
    })

### Celda A4 · Consolidación y actualización de tendencias_distritos.json

In [ ]:
tj = json.load(open('/content/tendencias_distritos.json', encoding='utf-8'))
por_id = {d['id']: d for d in tj['distritos']}
for did, g in df2.groupby(CAMPO_ID):
    d = por_id[str(did)]
    mapa = {f.strftime('%Y-%m'): (a, t) for f, a, t in zip(g['fecha'], g['aod'], g['t2m'])}
    for p in d['serie']:
        a, t = mapa.get(p['fecha'], (None, None))
        p['aod'] = None if a is None or pd.isna(a) else round(float(a), 3)
        p['t2m'] = None if t is None or pd.isna(t) else round(float(t), 2)
    d['tendencias']['aod'] = analizar(g[['fecha', 'aod']])
    d['tendencias']['t2m'] = analizar(g[['fecha', 't2m']])

tj['meta'].update({
    'fuente_aod': 'NASA MODIS MCD19A2 v061 MAIAC (AOD 550 nm, 1 km)',
    'fuente_t2m': 'NASA MERRA-2 (temperatura del aire a 2 m, ~50 km)',
    'periodo_aod': f'{AÑO_INI}-01 a {AÑO_FIN}-12',
    'periodo_t2m': f'{AÑO_INI}-01 a {AÑO_FIN}-12',
    'nota_t2m': 'MERRA-2 tiene celdas de unos 50 km: los distritos comparten casi el mismo valor. Sirve para la tendencia regional, no para comparar distritos.',
    'nota_aod': 'El AOD mide polvo y humo en toda la columna de aire, no lo que se respira a nivel de la calle.'
})
tj['meta']['unidades'].update({
    'aod': 'por año (adimensional)',
    't2m': '°C/año',
    'pm10': 'µg/m³ por año'
})
tj['estaciones'] = estaciones

json.dump(tj, open('/content/tendencias_distritos.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=2)
print('=== RESUMEN DE TENDENCIAS PARA ALBA ===')
tabla_resumen = pd.DataFrame([{
    'distrito': d['nombre'],
    'aod_sen': d['tendencias']['aod']['sen_pendiente_anual'],
    'aod_p': d['tendencias']['aod']['mk_p'],
    't2m_sen': d['tendencias']['t2m']['sen_pendiente_anual'],
    't2m_p': d['tendencias']['t2m']['mk_p']
} for d in tj['distritos']])
print(tabla_resumen.to_string())

from google.colab import files
files.download('/content/tendencias_distritos.json')
files.download('/content/series_aod_t2m_distritos.csv')